# TranscriptFormer cell embeddings — Open Problems batch correction

This notebook is the TranscriptFormer counterpart of `batch_corr_op.ipynb`.
It embeds the same four datasets with the human `tf-sapiens` checkpoint and
evaluates those embeddings with the same scIB batch-correction metrics.

Datasets included: `dkd`, `gtex_v9`, `hypomap`, and `mouse_pancreas_atlas`.
`immune_cell_atlas` and `tabula_sapiens` are intentionally excluded.


## Recreate the dedicated environment on Jean Zay

Run these commands once from the repository root on a Jean Zay login node.
They create an isolated Python 3.11 virtual environment, install the pinned
TranscriptFormer release and the notebook/benchmark dependencies, then register
the kernel used by Papermill.

```bash
module purge
module load python/3.11.5
source /etc/profile.d/proxy.sh

export WORK="${WORK:-/lustre/fswork/projects/rech/xeg/$USER}"
export TF_ENV="$WORK/venvs/transcriptformer-0.6.1"
export UV="$HOME/.local/bin/uv"
export UV_CACHE_DIR="$WORK/.cache/uv"
export XDG_CACHE_HOME="$WORK/.cache"
export MPLCONFIGDIR="$WORK/.cache/matplotlib"
export PYTHON_311="/gpfslocalsup/pub/anaconda-py3/2023.09/envs/python-3.11.5/bin/python"
mkdir -p "$WORK/venvs" "$UV_CACHE_DIR" "$MPLCONFIGDIR"
"$UV" venv --python "$PYTHON_311" "$TF_ENV"
source "$TF_ENV/bin/activate"

"$UV" pip install --python "$TF_ENV/bin/python"   "transcriptformer==0.6.1"   "torch==2.5.1"   "papermill>=2.6,<3"   "ipykernel>=6,<7"   scib-metrics scanpy leidenalg

"$TF_ENV/bin/python" -m ipykernel install --user   --name transcriptformer   --display-name "Python (TranscriptFormer 0.6.1)"

# The checkpoint must already exist before the notebook is submitted.
test -d "$WORK/models/transcriptformer/tf_sapiens"

# Record the exact environment for later reproduction.
"$TF_ENV/bin/python" --version
"$UV" pip freeze --python "$TF_ENV/bin/python" > "$WORK/venvs/transcriptformer-0.6.1.freeze.txt"
```

TranscriptFormer 0.6.1 pins PyTorch 2.5.1. Do not upgrade it independently:
the official documentation warns that PyTorch 2.6+ can cause checkpoint loading
errors. For these human datasets use `tf-sapiens`. The model expects raw,
unnormalised counts and Ensembl gene identifiers; the preparation cell below
checks both conditions before inference.


## GPU execution on Jean Zay

From the repository root, activate the dedicated environment inside the Slurm
command and execute this notebook in place. A V100 should start with batch size
1 and mixed precision; the notebook uses those conservative settings.

```bash
sbatch_tail   --ntasks-per-node=1   --gres=gpu:1   --time=04:00:00   --account=wbg@v100   --nodes=1   --partition=gpu_p2l   --hint=nomultithread   --signal=SIGUSR1@180   --qos=qos_gpu-t4   --cpus-per-task=4   slurm/any_sub.sh   "bash -lc 'module unload cuda/12.2.0 && source /etc/profile.d/proxy.sh && export IPYTHONDIR=$WORK/.cache/ipython && mkdir -p $WORK/.cache/ipython && source $WORK/venvs/transcriptformer-0.6.1/bin/activate && papermill --progress-bar --autosave-cell-every 120 -k transcriptformer --log-output notebooks/scPRINT-2-repro-notebooks/batch_corr_op_transcriptformer.ipynb notebooks/scPRINT-2-repro-notebooks/batch_corr_op_transcriptformer.ipynb'"
```

If block-mask compilation fails on the allocated GPU/software stack, add
`--disable-compile-block-mask` to `TF_INFERENCE_EXTRA_ARGS` below and rerun.
That switch is a troubleshooting fallback, not the default.


In [1]:
from pathlib import Path
import gc
import json
import os
import shutil
import subprocess

os.environ.update({
    "HF_HUB_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "WANDB_MODE": "offline",
    "WANDB_DISABLED": "true",
    "AWS_EC2_METADATA_DISABLED": "true",
})

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection


/lustre/fswork/projects/rech/xeg/uat95fg/venvs/transcriptformer-0.6.1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Fail early if Papermill is using the wrong environment or no GPU was allocated.
subprocess.run(["transcriptformer", "--help"], check=True, stdout=subprocess.DEVNULL)
subprocess.run(["nvidia-smi"], check=True)

import importlib.metadata as metadata

print("TranscriptFormer:", metadata.version("transcriptformer"))
print("Python executable:", shutil.which("python"))


Mon Aug  3 17:15:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-SXM2-32GB           On  |   00000000:16:00.0 Off |                    0 |
| N/A   33C    P0             45W /  300W |       0MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
WORK = Path(os.environ.get("WORK", Path.cwd()))
SCRATCH = Path(os.environ.get("SCRATCH", WORK))

DATA_ROOT = Path("data/temp")
TF_CACHE_ROOT = SCRATCH / "scprint_data"
PREPARED_ROOT = TF_CACHE_ROOT / "transcriptformer_inputs"
OUTPUT_ROOT = TF_CACHE_ROOT / "transcriptformer_outputs"
RESULT_ROOT = Path("data/results/transcriptformer")
CHECKPOINT_ROOT = WORK / "models/transcriptformer"
CHECKPOINT_PATH = CHECKPOINT_ROOT / "tf_sapiens"

for directory in (PREPARED_ROOT, OUTPUT_ROOT, RESULT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

TF_INFERENCE_EXTRA_ARGS = ["--disable-compile-block-mask"]

datasets = (
    "cellxgene_census/dkd",
    "cellxgene_census/gtex_v9",
    "cellxgene_census/hypomap",
    "cellxgene_census/mouse_pancreas_atlas",
)

assert "cellxgene_census/immune_cell_atlas" not in datasets
assert "cellxgene_census/tabula_sapiens" not in datasets


## Prepare raw-count AnnData inputs

The official TranscriptFormer loader reads `.raw.X` first when it exists. To
make the input unambiguous, this notebook writes a temporary AnnData whose `X`
is the selected raw-count matrix and whose `var["ensembl_id"]` contains Ensembl
IDs. It also samples the matrix to reject negative or non-integer-like values.

Prepared inputs and model outputs are cached, so a restarted Slurm job resumes
at the first unfinished dataset instead of recomputing completed inference.


In [4]:
def _sample_values(matrix, n_rows=128):
    sample = matrix[: min(n_rows, matrix.shape[0])]
    values = sample.data if sp.issparse(sample) else np.asarray(sample).ravel()
    return np.asarray(values)


def _add_ensembl_ids(adata):
    if "ensembl_id" in adata.var:
        ids = adata.var["ensembl_id"].astype(str)
    elif "feature_id" in adata.var:
        ids = adata.var["feature_id"].astype(str)
    elif pd.Index(adata.var_names.astype(str)).str.startswith("ENS").all():
        ids = pd.Series(adata.var_names.astype(str), index=adata.var_names)
    else:
        raise ValueError(
            "No Ensembl IDs found: expected var['ensembl_id'], "
            "var['feature_id'], or Ensembl-formatted var_names."
        )
    adata.var["ensembl_id"] = ids.to_numpy()


def prepare_transcriptformer_input(name):
    slug = name.rsplit("/", 1)[-1]
    source_path = DATA_ROOT / f"{name}.h5ad"
    processed_path = DATA_ROOT / f"{name}_proc.h5ad"
    prepared_path = PREPARED_ROOT / f"{slug}_raw_counts.h5ad"
    if prepared_path.exists():
        return prepared_path

    if source_path.exists():
        source = sc.read_h5ad(source_path)
        prepared = source.raw.to_adata() if source.raw is not None else source
    elif processed_path.exists():
        print(f"Using cached processed counts from {processed_path}", flush=True)
        source = sc.read_h5ad(processed_path)
        prepared = source
    else:
        raise FileNotFoundError(
            f"{name}: expected {source_path} or {processed_path}; "
            "network downloads are disabled in this notebook."
        )
    prepared.obs = source.obs.copy()
    obs_name_key = "_transcriptformer_input_obs_name"
    if obs_name_key in prepared.obs:
        raise ValueError(f"{name}: reserved obs column already exists: {obs_name_key}")
    prepared.obs[obs_name_key] = prepared.obs_names.astype(str)
    _add_ensembl_ids(prepared)

    values = _sample_values(prepared.X)
    if values.size and (
        np.nanmin(values) < 0
        or not np.allclose(values, np.rint(values), rtol=0, atol=1e-6)
    ):
        raise ValueError(
            f"{name}: TranscriptFormer requires raw, non-negative integer counts."
        )

    # Avoid a second, potentially normalized matrix taking precedence at inference.
    prepared.raw = None
    prepared.write_h5ad(prepared_path, compression="lzf")
    del source, prepared
    gc.collect()
    return prepared_path


In [5]:
if not CHECKPOINT_PATH.is_dir():
    raise FileNotFoundError(
        f"Missing checkpoint directory: {CHECKPOINT_PATH}. "
        "Network downloads are disabled in this notebook."
    )

print("Checkpoint:", CHECKPOINT_PATH)


Checkpoint: /lustre/fswork/projects/rech/xeg/uat95fg/models/transcriptformer/tf_sapiens


## Run TranscriptFormer and the scIB benchmark

Cell embeddings are read from `obsm["embeddings"]`, the official output field.
The notebook verifies exact cell-name/order preservation before assigning the
embedding to the benchmark key. Scores are written after every dataset.


In [6]:
metrics = {}

for name in datasets:
    slug = name.rsplit("/", 1)[-1]
    print(f"doing {name}", flush=True)

    prepared_path = prepare_transcriptformer_input(name)
    output_path = OUTPUT_ROOT / f"{slug}_tf_sapiens_embeddings.h5ad"
    score_path = RESULT_ROOT / f"{slug}_scib.csv"

    if not output_path.exists():
        command = [
            "transcriptformer",
            "inference",
            "--checkpoint-path",
            str(CHECKPOINT_PATH),
            "--data-file",
            str(prepared_path),
            "--gene-col-name",
            "ensembl_id",
            "--use-raw",
            "False",
            "--output-path",
            str(OUTPUT_ROOT),
            "--output-filename",
            output_path.name,
            "--emb-type",
            "cell",
            "--device",
            "cuda",
            "--num-gpus",
            "1",
            "--precision",
            "16-mixed",
            "--batch-size",
            "1",
            "--oom-dataloader",
            "--n-data-workers",
            "2",
            *TF_INFERENCE_EXTRA_ARGS,
        ]
        print("Running:", " ".join(command), flush=True)
        subprocess.run(command, check=True)

    embedded = ad.read_h5ad(output_path)
    prepared_backed = ad.read_h5ad(prepared_path, backed="r")
    prepared_obs_names = prepared_backed.obs_names.astype(str).to_numpy(copy=True)
    prepared_backed.file.close()
    obs_name_key = "_transcriptformer_input_obs_name"
    if obs_name_key not in embedded.obs:
        raise KeyError(f"{name}: output obs is missing {obs_name_key!r}")
    embedded_obs_names = embedded.obs.pop(obs_name_key).astype(str).to_numpy()
    if not np.array_equal(embedded_obs_names, prepared_obs_names):
        raise RuntimeError(
            f"{name}: TranscriptFormer output cell names/order differ from input; "
            "refusing to align embeddings positionally."
        )
    embedded.obs_names = embedded_obs_names
    if "embeddings" not in embedded.obsm:
        raise KeyError(f"{name}: output has no obsm['embeddings'] field")
    for required_obs in ("donor_id", "cell_type"):
        if required_obs not in embedded.obs:
            raise KeyError(f"{name}: output obs is missing {required_obs!r}")

    embedded.obsm["transcriptformer_emb"] = np.asarray(
        embedded.obsm["embeddings"], dtype=np.float32
    )
    benchmark = Benchmarker(
        embedded,
        batch_key="donor_id",
        label_key="cell_type",
        embedding_obsm_keys=["transcriptformer_emb"],
        bio_conservation_metrics=BioConservation(),
        batch_correction_metrics=BatchCorrection(),
        n_jobs=4,
    )
    benchmark.benchmark()
    result = benchmark.get_results(min_max_scale=False)
    result.to_csv(score_path)
    metrics[name] = result
    display(result)

    del embedded, benchmark
    gc.collect()


doing cellxgene_census/dkd


Running: transcriptformer inference --checkpoint-path /lustre/fswork/projects/rech/xeg/uat95fg/models/transcriptformer/tf_sapiens --data-file /lustre/fsn1/projects/rech/xeg/uat95fg/scprint_data/transcriptformer_inputs/dkd_raw_counts.h5ad --gene-col-name ensembl_id --use-raw False --output-path /lustre/fsn1/projects/rech/xeg/uat95fg/scprint_data/transcriptformer_outputs --output-filename dkd_tf_sapiens_embeddings.h5ad --emb-type cell --device cuda --num-gpus 1 --precision 16-mixed --batch-size 1 --oom-dataloader --n-data-workers 2 --disable-compile-block-mask


2026-08-03 17:15:54,079 - INFO - Loading vocabulary file: /lustre/fswork/projects/rech/xeg/uat95fg/models/transcriptformer/tf_sapiens/vocabs/assay
2026-08-03 17:15:54,133 - INFO - Loading ESM2 mappings from /lustre/fswork/projects/rech/xeg/uat95fg/models/transcriptformer/tf_sapiens/vocabs


2026-08-03 17:16:03,825 - INFO - Building gene vocabulary


2026-08-03 17:16:04,491 - INFO - Instantiating Transcriptformer model
2026-08-03 17:16:04,492 - INFO - Instantiating the transcriptformer model


...

Predicting DataLoader 0:  69%|██████▉   | 27196/39176 [1:19:39<35:05,  5.69it/s]

Predicting DataLoader 0:  69%|██████▉   | 27198/39176 [1:19:40<35:05,  5.69it/s]

Predicting DataLoader 0:  69%|██████▉   | 27200/39176 [1:19:40<35:04,  5.69it/s]

Predicting DataLoader 0:  69%|██████▉   | 27202/39176 [1:19:40<35:04,  5.69it/s]

Predicting DataLoader 0:  69%|██████▉   | 27204/39176 [1:19:41<35:04,  5.69it/s]

In [ ]:
combined = pd.concat(metrics, names=["dataset", "method"])
combined.to_csv(RESULT_ROOT / "transcriptformer_scib_all_datasets.csv")
combined
